In [3]:
import os
import re
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
from config import RAW_DATA_DIR
from src.services.bloomberg_client import BloombergClient
from src.services.mail_client import OutlookReader, SMTPClient, IMAPReader

In [10]:
with BloombergClient() as myclient:
    data = myclient.BDH(["GIB CN 01/16/2026 C130 Equity"], ["PX_MID","PX_LAST"],"20251231","20260108")
    data['PX_LAST'] = data['PX_LAST'].fillna(data['PX_MID'])
    print(data.iloc[:,-8:])

                          Ticker        Date  PX_MID  PX_LAST
0  GIB CN 01/16/2026 C130 Equity  2025-12-31    0.85     0.85
1  GIB CN 01/16/2026 C130 Equity  2026-01-02    0.44     0.44
2  GIB CN 01/16/2026 C130 Equity  2026-01-05    0.38     0.38
3  GIB CN 01/16/2026 C130 Equity  2026-01-06    0.90     0.60
4  GIB CN 01/16/2026 C130 Equity  2026-01-07    0.55     0.55
5  GIB CN 01/16/2026 C130 Equity  2026-01-08    2.05     1.30


In [ ]:
# =============================================================================
# Extract CSV attachments from Outlook emails via LOCAL APP (no network)
# =============================================================================
# Uses OutlookReader (direct app access, firewall-proof)
# =============================================================================

pattern = re.compile(r"All_Positions(\d{8})\.csv$")
output_dir = Path(RAW_DATA_DIR) / "all_positions"
output_dir.mkdir(parents=True, exist_ok=True)
saved = 0

try:
    outlook_reader = OutlookReader()
    for outlook_item in outlook_reader.get_emails_from_folder(folder_name="Confirmations", limit=10):
        # outlook_item is a native Outlook mail object with .Attachments
        for attachment in outlook_item.Attachments:
            filename = attachment.FileName
            if filename and pattern.match(filename):
                date_str = pattern.match(filename).group(1)
                if 20240101 <= int(date_str) <= 20260319:
                    attachment.SaveAsFile(str(output_dir / filename))
                    saved += 1
                    print(f"Saved: {filename}")
except Exception as e:
    print(f"Error: {e}")

print(f"\nTotal saved: {saved} CSV files")

           Ticker        Date  PX_LAST
0  AAPL US Equity  2026-01-30   259.48
1  AAPL US Equity  2026-02-27   264.18
